<a href="https://colab.research.google.com/github/llayan-1/typiclust-cw2/blob/main/typiclust_cw2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [74]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Subset
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.cluster import KMeans
from collections import defaultdict

import random
import numpy as np


In [61]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

test_set = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

In [62]:
# Make results reproducible
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

num_train = len(train_set)
all_indices = list(range(num_train))

# Initial labelled set size
initial_labelled_size = 100

# Shuffle indices
random.shuffle(all_indices)

# Split into labelled and unlabelled pools
L_indices = all_indices[:initial_labelled_size]
U_indices = all_indices[initial_labelled_size:]

print("Initial labelled pool size:", len(L_indices))
print("Unlabelled pool size:", len(U_indices))

Initial labelled pool size: 100
Unlabelled pool size: 49900


In [63]:
labelled_subset = Subset(train_set, L_indices)

print("Labelled subset created:", len(labelled_subset))

Labelled subset created: 100


In [64]:
batch_size = 32

labelled_loader = DataLoader(labelled_subset, batch_size=batch_size,
                             shuffle=True)

test_loader = DataLoader(test_set,batch_size=batch_size,
                         shuffle=False)

print("Labelled loader and test loader are ready.")

Labelled loader and test loader are ready.


In [65]:
unlabelled_subset = Subset(train_set, U_indices)

unlabelled_loader = DataLoader(unlabelled_subset,batch_size=batch_size,
    shuffle=False)

print("Unlabelled subset created:", len(unlabelled_subset))
print("Unlabelled loader is ready.")

Unlabelled subset created: 49900
Unlabelled loader is ready.


In [66]:
class SmallCNN(nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc_embed = nn.Linear(256, embedding_dim)
        self.fc_out = nn.Linear(embedding_dim, 10)

    def forward(self, x, return_embedding=False):
        x = self.pool(F.relu(self.conv1(x)))   # 32x32 -> 16x16
        x = self.pool(F.relu(self.conv2(x)))   # 16x16 -> 8x8
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        embedding = self.fc_embed(x)
        logits = self.fc_out(embedding)

        if return_embedding:
            return logits, embedding
        return logits

In [67]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SmallCNN().to(device)

print(model)

SmallCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=4096, out_features=256, bias=True)
  (fc_embed): Linear(in_features=256, out_features=128, bias=True)
  (fc_out): Linear(in_features=128, out_features=10, bias=True)
)


In [68]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [69]:
def train_model(model, loader, epochs=5):

    model.train()

    for epoch in range(epochs):

        running_loss = 0.0

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {running_loss/len(loader):.4f}")

In [70]:
train_model(model, labelled_loader, epochs=5)

Epoch 1, Loss: 2.2550
Epoch 2, Loss: 2.2144
Epoch 3, Loss: 2.1133
Epoch 4, Loss: 1.8985
Epoch 5, Loss: 1.7496


In [71]:
def extract_embeddings(model, loader):

    model.eval()

    embeddings = []
    indices = []

    with torch.no_grad():

        for i, (images, _) in enumerate(loader):

            images = images.to(device)

            logits, embed = model(images, return_embedding=True)

            embeddings.append(embed.cpu())

    embeddings = torch.cat(embeddings)

    return embeddings

In [72]:
U_embeddings = extract_embeddings(model, unlabelled_loader)
print("Embedding matrix shape:", U_embeddings.shape)

U_embeddings_np = U_embeddings.numpy()
print(U_embeddings_np.shape)

num_clusters = 100

Embedding matrix shape: torch.Size([49900, 128])
(49900, 128)


In [73]:
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(U_embeddings_np)

print("Number of cluster assignments:", len(cluster_labels))
print("Unique clusters found:", len(np.unique(cluster_labels)))

Number of cluster assignments: 49900
Unique clusters found: 100


In [75]:
cluster_to_positions = defaultdict(list)

# group positions by cluster
for pos, cluster_id in enumerate(cluster_labels):
    cluster_to_positions[cluster_id].append(pos)

selected_positions = []

for cluster_id, positions in cluster_to_positions.items():

    cluster_center = kmeans.cluster_centers_[cluster_id]
    cluster_points = U_embeddings_np[positions]

    distances = np.linalg.norm(cluster_points - cluster_center, axis=1)

    best_local_index = np.argmin(distances)
    best_position = positions[best_local_index]

    selected_positions.append(best_position)

print("Selected samples:", len(selected_positions))

Selected samples: 100


In [76]:
selected_dataset_indices = [U_indices[pos] for pos in selected_positions]

print("First few selected dataset indices:", selected_dataset_indices[:10])
# add to labelled pool
L_indices.extend(selected_dataset_indices)

# remove from unlabelled pool
selected_set = set(selected_positions)
U_indices = [idx for pos, idx in enumerate(U_indices) if pos not in selected_set]

print("Updated labelled size:", len(L_indices))
print("Updated unlabeled size:", len(U_indices))

First few selected dataset indices: [21230, 31191, 24229, 9188, 33435, 21789, 10701, 13714, 31549, 27675]
Updated labelled size: 200
Updated unlabeled size: 49800


In [77]:
labelled_subset = Subset(train_set, L_indices)

labelled_loader = DataLoader(
    labelled_subset,
    batch_size=32,
    shuffle=True
)

print("New labelled dataset size:", len(labelled_subset))

New labelled dataset size: 200


In [78]:
model = SmallCNN().to(device)

train_model(model, labelled_loader, epochs=5)

Epoch 1, Loss: 2.3001
Epoch 2, Loss: 2.2990
Epoch 3, Loss: 2.3006
Epoch 4, Loss: 2.2995
Epoch 5, Loss: 2.2976


In [82]:
unlabelled_subset = Subset(train_set, U_indices)

unlabelled_loader = DataLoader(
    unlabelled_subset,
    batch_size=32,
    shuffle=False
)

print("New unlabelled dataset size:", len(unlabelled_subset))

New unlabelled dataset size: 49800


In [83]:
U_embeddings = extract_embeddings(model, unlabelled_loader)
print("Updated embedding matrix shape:", U_embeddings.shape)

Updated embedding matrix shape: torch.Size([49800, 128])


In [84]:
U_embeddings_np = U_embeddings.numpy()

num_clusters = 100

kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(U_embeddings_np)

print("Number of cluster assignments:", len(cluster_labels))
print("Unique clusters found:", len(np.unique(cluster_labels)))

Number of cluster assignments: 49800
Unique clusters found: 100


In [85]:
cluster_to_positions = defaultdict(list)

for pos, cluster_id in enumerate(cluster_labels):
    cluster_to_positions[cluster_id].append(pos)

selected_positions = []

for cluster_id, positions in cluster_to_positions.items():
    cluster_center = kmeans.cluster_centers_[cluster_id]
    cluster_points = U_embeddings_np[positions]

    distances = np.linalg.norm(cluster_points - cluster_center, axis=1)

    best_local_index = np.argmin(distances)
    best_position = positions[best_local_index]

    selected_positions.append(best_position)

print("Selected samples:", len(selected_positions))

Selected samples: 100


In [86]:
selected_dataset_indices = [U_indices[pos] for pos in selected_positions]

L_indices.extend(selected_dataset_indices)

selected_set = set(selected_positions)
U_indices = [idx for pos, idx in enumerate(U_indices) if pos not in selected_set]

print("Updated labelled size:", len(L_indices))
print("Updated unlabelled size:", len(U_indices))

Updated labelled size: 300
Updated unlabelled size: 49700


In [88]:
def active_learning_round(
    train_set,
    L_indices,
    U_indices,
    batch_size=32,
    epochs=5,
    num_clusters=100
):
    # 1. Build subsets and loaders
    labelled_subset = Subset(train_set, L_indices)
    unlabelled_subset = Subset(train_set, U_indices)

    labelled_loader = DataLoader(labelled_subset, batch_size=batch_size, shuffle=True)
    unlabelled_loader = DataLoader(unlabelled_subset, batch_size=batch_size, shuffle=False)

    # 2. Create and train model
    model = SmallCNN().to(device)
    train_model(model, labelled_loader, epochs=epochs)

    # 3. Extract embeddings from unlabelled pool
    U_embeddings = extract_embeddings(model, unlabelled_loader)
    U_embeddings_np = U_embeddings.numpy()

    # 4. Cluster embeddings
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(U_embeddings_np)

    # 5. Select one representative per cluster
    from collections import defaultdict
    cluster_to_positions = defaultdict(list)

    for pos, cluster_id in enumerate(cluster_labels):
        cluster_to_positions[cluster_id].append(pos)

    selected_positions = []

    for cluster_id, positions in cluster_to_positions.items():
        cluster_center = kmeans.cluster_centers_[cluster_id]
        cluster_points = U_embeddings_np[positions]

        distances = np.linalg.norm(cluster_points - cluster_center, axis=1)

        best_local_index = np.argmin(distances)
        best_position = positions[best_local_index]

        selected_positions.append(best_position)

    # 6. Convert selected positions back to original dataset indices
    selected_dataset_indices = [U_indices[pos] for pos in selected_positions]

    # 7. Update pools
    L_indices.extend(selected_dataset_indices)

    selected_set = set(selected_positions)
    U_indices = [idx for pos, idx in enumerate(U_indices) if pos not in selected_set]

    return L_indices, U_indices

In [89]:
# Reset pools
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

all_indices = list(range(len(train_set)))
random.shuffle(all_indices)

initial_labelled_size = 100
L_indices = all_indices[:initial_labelled_size]
U_indices = all_indices[initial_labelled_size:]

print("Start -> Labelled:", len(L_indices), "| Unlabelled:", len(U_indices))

num_rounds = 3

for round_num in range(num_rounds):
    print(f"\n--- Round {round_num + 1} ---")
    L_indices, U_indices = active_learning_round(
        train_set,
        L_indices,
        U_indices,
        batch_size=32,
        epochs=5,
        num_clusters=100
    )
    print("After round", round_num + 1, "-> Labelled:", len(L_indices), "| Unlabelled:", len(U_indices))

Start -> Labelled: 100 | Unlabelled: 49900

--- Round 1 ---
Epoch 1, Loss: 2.3032
Epoch 2, Loss: 2.3031
Epoch 3, Loss: 2.3021
Epoch 4, Loss: 2.2974
Epoch 5, Loss: 2.3032
After round 1 -> Labelled: 200 | Unlabelled: 49800

--- Round 2 ---
Epoch 1, Loss: 2.3013
Epoch 2, Loss: 2.3016
Epoch 3, Loss: 2.3007
Epoch 4, Loss: 2.2978
Epoch 5, Loss: 2.2992
After round 2 -> Labelled: 300 | Unlabelled: 49700

--- Round 3 ---
Epoch 1, Loss: 2.3005
Epoch 2, Loss: 2.2988
Epoch 3, Loss: 2.2998
Epoch 4, Loss: 2.3002
Epoch 5, Loss: 2.3003
After round 3 -> Labelled: 400 | Unlabelled: 49600
